# MAMA-MIA — 10: Rezolyutsiya doza-javobi

## Nega bu kerak

09-notebook kattalashtirishni o'lchadi va ikki narsani topdi:

1. **Ko'rinadigan masshtab mexanizm emas.** NACT'da (92/128 toza holat)
   chegara xatosi millimetrda doimiy (±10%), pikselda esa k ga
   proporsional. Kattalashtirish segmentatsiyani yaxshilamaydi.
2. **Ko'rish maydoni esa halokatli ahamiyatga ega.** DUKE k=2 da
   holatlarning 79.7% ida o'sma kadrdan chiqib ketgan; saqlanish bilan
   Dice o'zgarishi ρ = +0.755 (p = 1e-80), o'sma to'rtdan birdan kam
   saqlansa Dice −0.470 tushadi.

Lekin 09 da kattalashtirish, ko'rish maydoni va rezolyutsiya birga
o'zgargan edi. Qolgan savol: **rezolyutsiyaning o'zi ahamiyatlimi?**

## Usul — faqat rezolyutsiya

Tasvir r koeffitsientiga siqiladi, keyin yana 192 ga qaytariladi:

```
192 → (192·r) → 192
```

Natijada:

| o'zgaradi | o'zgarmaydi |
|---|---|
| haqiqiy rezolyutsiya (ma'lumot yo'qoladi) | ko'rish maydoni |
| | ko'rinadigan masshtab |
| | piksel soni |
| | ground truth — **umuman tegilmaydi** |

**Maska siqilmaydi.** Vazifa aynan o'sha bo'lib qoladi, faqat kirishdagi
ma'lumot kamayadi. Shuning uchun bu 09 dan ham toza manipulyatsiya:
qirqish ham, masshtab siljishi ham yo'q.

| r | effektiv qadam |
|---|---|
| 1.00 | 1.5 mm (o'zgarishsiz) |
| 0.75 | 2.0 mm |
| 0.50 | 3.0 mm |
| 0.33 | 4.5 mm |

Bu diapazon ataylab tanlangan: V0 DUKE'ni native 0.804 mm dan
1.876 mm ga tushiradi, ya'ni ~2.3× — r=0.5 va r=0.33 shuni qamrab oladi.

## Oldindan qayd etilgan taxmin

(09 dagi taxminim rad etilgan edi; bu safar ham yozib qo'yaman.)

1. Dice qo'polroq rezolyutsiyada **tushadi** — kattalashtirishdan farqli
   o'laroq, bu yerda haqiqiy chegara ma'lumoti yo'qoladi.
2. Tushish 1.5 → 3.0 mm oralig'ida **mo'tadil** (~0.03 dan kam), chunki
   o'sma chegarasi DCE-MRI da allaqachon noaniq.
3. Ikkala foldda ham yo'nalish bir xil.

**Agar Dice 4.5 mm gacha ham sezilarli tushmasa** — rezolyutsiya ham
mexanizm emas. Unda manbalararo V0 effektining sababi masshtabda ham,
rezolyutsiyada ham, ko'rish maydonida ham emas; qoladigan nomzod —
piksel fazosidagi domen o'xshashligi. Bu ham aniq javob.



## 1. Sozlamalar

In [ ]:
import os, gc, json, time, random, glob, collections
import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import zoom as ndzoom, binary_erosion
from scipy.ndimage import distance_transform_edt as edt

RES      = [1.0, 0.75, 0.5, 0.33]   # rezolyutsiya koeffitsienti
HELD_OUT = ["NACT", "DUKE"]
SEEDS    = [42, 1]

IMG_SIZE, BATCH, LR = 192, 16, 3e-4
EPOCHS, EPOCH_SLICES, EVAL_EVERY = 50, 12000, 10
VAL_FRAC, POS_FRAC = 0.2, 0.7
SPACING = 1.5                        # to'r hech qachon o'zgarmaydi

if torch.cuda.is_available():
    dev, AMP, NUM_WORKERS, PIN = torch.device("cuda"), True, 2, True
    name = torch.cuda.get_device_name(0)
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    dev, AMP, NUM_WORKERS, PIN = torch.device("mps"), False, 0, False
    name = "Apple Silicon (MPS)"
else:
    dev, AMP, NUM_WORKERS, PIN = torch.device("cpu"), False, 0, False
    name = "CPU"
print(f"Qurilma: {dev}  {name}   AMP={AMP}  workers={NUM_WORKERS}")

OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
CKPT = os.path.join(OUT, "ressweep")
os.makedirs(CKPT, exist_ok=True)
print("Natijalar:", os.path.abspath(CKPT))
print(f"\nProtokol: r={RES}, fold={HELD_OUT}, seed={SEEDS} "
      f"→ {len(RES)*len(HELD_OUT)*len(SEEDS)} run")
print("Effektiv qadam:", ", ".join(f"r={r:g}→{SPACING/r:.2f}mm" for r in RES))

## 2. Ma'lumot

In [ ]:
DATA_DIR = os.environ.get("MAMAMIA_V2", "")

cand = sorted({os.path.dirname(p) for p in
               glob.glob("/kaggle/input/**/V2_iso_scale/mamamia_*.h5", recursive=True)})
if DATA_DIR:
    cand = [DATA_DIR] + cand
assert cand, ("V2_iso_scale topilmadi. Kaggle'da variantlar notebook'ini Input "
              "qiling; mahalliy ishlatsangiz MAMAMIA_V2 ni papkaga qo'ying.")
DATA = cand[0]
print("V2:", DATA)

def case_of(k):
    return k[:-2] if k.endswith(("_L","_R")) else k

INDEX = []
for p in sorted(glob.glob(os.path.join(DATA, "mamamia_*.h5"))):
    src = os.path.basename(p)[len("mamamia_"):-len(".h5")]
    with h5py.File(p, "r") as h:
        for key in h:
            mk = h[key]["msk"]
            assert mk.shape[0] == IMG_SIZE == mk.shape[1]
            nz = np.where(np.asarray(mk).any(axis=(0,1)))[0]
            INDEX.append({"src":src,"file":p,"key":key,"case":case_of(key),
                          "n":int(mk.shape[2]),"pos":nz.tolist()})
print(f"{len(INDEX)} namuna, {len({r['case'] for r in INDEX})} holat")
print(collections.Counter(r["src"] for r in INDEX))

## 3. Rezolyutsiyani pasaytirish

Faqat **tasvir** siqilib qaytariladi. Maska tegilmaydi — shuning uchun
ground truth hamma r da aynan bir xil bo'lishi kerak, va buni
`gt_vox` ning o'zgarmasligi tekshiradi.

In [ ]:
def fit_center(a, pad_value):
    H, W = a.shape
    if H == IMG_SIZE and W == IMG_SIZE:
        return a
    out = np.full((IMG_SIZE, IMG_SIZE), pad_value, a.dtype)
    hs, ws = min(H, IMG_SIZE), min(W, IMG_SIZE)
    sy, sx = (H - hs) // 2, (W - ws) // 2
    dy, dx = (IMG_SIZE - hs) // 2, (IMG_SIZE - ws) // 2
    out[dy:dy+hs, dx:dx+ws] = a[sy:sy+hs, sx:sx+ws]
    return out


def degrade(img, r):
    """192 -> 192*r -> 192. Ko'rish maydoni va masshtab saqlanadi."""
    if r >= 0.999:
        return img
    small = ndzoom(img, r, order=1, mode="nearest")
    back = ndzoom(small, (IMG_SIZE / small.shape[0], IMG_SIZE / small.shape[1]),
                  order=1, mode="nearest")
    return fit_center(back, 0.0)


def sharpness(a):
    """O'rtacha gradient kattaligi — rezolyutsiya haqiqatan tushganini tekshiradi."""
    gy, gx = np.gradient(a.astype(np.float32))
    return float(np.mean(np.hypot(gy, gx)))


def dice(a, b):
    s = a.sum() + b.sum()
    return (2.0 * np.logical_and(a, b).sum() / s) if s else float("nan")


def surf(M):
    return M & ~binary_erosion(M, iterations=1, border_value=0)


def assd(P, G, sampling):
    sp, sg = surf(P), surf(G)
    if not sp.any() or not sg.any():
        return float("nan")
    dg = edt(~sg, sampling=sampling)
    dp = edt(~sp, sampling=sampling)
    return float((dg[sp].mean() + dp[sg].mean()) / 2.0)


# --- Sinov: shakl saqlanadimi, o'tkirlik tushadimi ---
_r = np.random.default_rng(0)
_i = _r.normal(size=(IMG_SIZE, IMG_SIZE)).astype(np.float32)
_i[60:130, 60:130] += 3.0                      # keskin chekkali blok
print("r ning tasvirga ta'siri:")
for r in RES:
    a = degrade(_i, r)
    assert a.shape == (IMG_SIZE, IMG_SIZE), f"shakl buzildi: {a.shape}"
    print(f"  r={r:<5} qadam {SPACING/r:>4.2f} mm   shakl {a.shape}   "
          f"o'tkirlik {sharpness(a):.3f}   sigma {a.std():.3f}")
print("  (o'tkirlik r bilan birga tushishi kerak — ma'lumot yo'qolyapti)")

_s = [sharpness(degrade(_i, r)) for r in RES]
assert all(_s[i] >= _s[i+1] for i in range(len(_s)-1)), \
    "o'tkirlik monoton tushmadi — degrade() tekshirilsin"

_G = np.zeros((40, 40, 10), bool); _G[10:30, 10:30, 2:8] = True
assert assd(_G, _G, (SPACING,)*3) == 0.0

## 4. Dataset va model

In [ ]:
NORM = {}

class SliceDS(Dataset):
    def __init__(self, items, train, r, epoch_len=None):
        self.items, self.train, self.r = items, train, r
        self.h = {}
        if train:
            self.epoch_len = epoch_len or EPOCH_SLICES
            self.pos = [(i,z) for i,rr in enumerate(items) for z in rr["pos"]]
            self.all = [(i,z) for i,rr in enumerate(items) for z in range(rr["n"])]
        else:
            self.flat = [(i,z) for i,rr in enumerate(items) for z in range(rr["n"])]
    def __len__(self): return self.epoch_len if self.train else len(self.flat)
    def _f(self,p):
        if p not in self.h: self.h[p]=h5py.File(p,"r")
        return self.h[p]
    def _st(self,rec):
        key=rec["key"]
        if key not in NORM:
            g=self._f(rec["file"])[key]; st=max(1,g["img"].shape[2]//8)
            sub=np.asarray(g["img"][:,:,::st],dtype=np.float32)
            lo,hi=np.percentile(sub,[1,99])
            if hi<=lo: lo,hi=float(sub.min()),float(sub.max())+1e-3
            c=np.clip(sub,lo,hi)
            NORM[key]=(float(lo),float(hi),float(c.mean()),float(c.std()+1e-6))
        return NORM[key]
    def _get(self,i,z):
        rec=self.items[i]; g=self._f(rec["file"])[rec["key"]]
        img=np.asarray(g["img"][:,:,z],dtype=np.float32)
        msk=np.asarray(g["msk"][:,:,z],dtype=np.float32)   # MASKA TEGILMAYDI
        lo,hi,mu,sd=self._st(rec)
        img=((np.clip(img,lo,hi)-mu)/sd).astype(np.float32)
        return degrade(img, self.r), msk
    def __getitem__(self,i):
        if self.train:
            pool=self.pos if (random.random()<POS_FRAC and self.pos) else self.all
            j,z=pool[random.randrange(len(pool))]
            img,msk=self._get(j,z)
            if random.random()<0.5: img,msk=img[::-1].copy(),msk[::-1].copy()
            if random.random()<0.3: img=img*random.uniform(0.9,1.1)+random.uniform(-0.1,0.1)
        else:
            j,z=self.flat[i]; img,msk=self._get(j,z)
        return torch.from_numpy(np.ascontiguousarray(img))[None], \
               torch.from_numpy(np.ascontiguousarray(msk))[None]
    def __del__(self):
        for f in getattr(self,"h",{}).values():
            try: f.close()
            except Exception: pass

def block(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1,bias=False),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1,bias=False),nn.BatchNorm2d(o),nn.ReLU(True))
class UNet(nn.Module):
    def __init__(self, ch=(32,64,128,256,512)):
        super().__init__()
        self.enc=nn.ModuleList(); c=1
        for k in ch[:-1]: self.enc.append(block(c,k)); c=k
        self.bottom=block(ch[-2],ch[-1])
        self.up=nn.ModuleList(); self.dec=nn.ModuleList()
        for k in reversed(ch[:-1]):
            self.up.append(nn.ConvTranspose2d(k*2,k,2,stride=2)); self.dec.append(block(k*2,k))
        self.head=nn.Conv2d(ch[0],1,1)
    def forward(self,x):
        sk=[]
        for e in self.enc: x=e(x); sk.append(x); x=F.max_pool2d(x,2)
        x=self.bottom(x)
        for u,d,s in zip(self.up,self.dec,reversed(sk)): x=d(torch.cat([u(x),s],1))
        return self.head(x)

bce=nn.BCEWithLogitsLoss()
def dice_loss(l,y,eps=1.0):
    p=torch.sigmoid(l); n=2*(p*y).sum((1,2,3))+eps; d=p.sum((1,2,3))+y.sum((1,2,3))+eps
    return 1-(n/d).mean()
def criterion(l,y): return 0.5*bce(l,y)+0.5*dice_loss(l,y)
print(f"model: {sum(p.numel() for p in UNet().parameters())/1e6:.2f}M parametr")

## 5. Ishga tushirish

In [ ]:
def split_cases(tc, seed, frac=VAL_FRAC):
    cs = sorted(tc); rng = random.Random(seed); rng.shuffle(cs)
    nv = max(1, int(len(cs) * frac))
    return set(cs[nv:]), set(cs[:nv])


def _fwd(model, x):
    if AMP:
        with torch.autocast("cuda", enabled=True):
            return model(x)
    return model(x)


@torch.no_grad()
def val_dice(model, items, r, bs=32):
    model.eval(); ds = SliceDS(items, False, r); vals = []
    for i, rec in enumerate(items):
        inter = union = 0.0
        for s in range(0, rec["n"], bs):
            zz = list(range(s, min(s + bs, rec["n"])))
            xs, ys = zip(*(ds._get(i, z) for z in zz))
            x = torch.from_numpy(np.stack(xs))[:, None].to(dev)
            y = torch.from_numpy(np.stack(ys))[:, None].to(dev)
            p = (torch.sigmoid(_fwd(model, x)) > 0.5).float()
            inter += (p * y).sum().item(); union += p.sum().item() + y.sum().item()
        vals.append(2 * inter / union if union else np.nan)
    del ds
    return float(np.nanmean(vals))


@torch.no_grad()
def evaluate(model, items, r, thr=0.5, bs=32):
    model.eval(); ds = SliceDS(items, False, r)
    sp = (SPACING,) * 3
    out = {}
    for i, rec in enumerate(items):
        P = np.zeros((IMG_SIZE, IMG_SIZE, rec["n"]), bool)
        G = np.zeros((IMG_SIZE, IMG_SIZE, rec["n"]), bool)
        sh = []
        for s in range(0, rec["n"], bs):
            zz = list(range(s, min(s + bs, rec["n"])))
            xs, ys = zip(*(ds._get(i, z) for z in zz))
            x = torch.from_numpy(np.stack(xs))[:, None].to(dev)
            p = (torch.sigmoid(_fwd(model, x)) > thr).float()
            P[:, :, zz] = p.squeeze(1).permute(1, 2, 0).cpu().numpy() > 0.5
            G[:, :, zz] = np.stack(ys, axis=-1) > 0.5
        zc = rec["n"] // 2
        a = ds._get(i, zc)[0]
        out[rec["key"]] = {"dice": dice(P, G), "assd_mm": assd(P, G, sp),
                           "gt_vox": int(G.sum()), "pred_vox": int(P.sum()),
                           "img_std": float(a.std()), "sharp": sharpness(a)}
    del ds
    return out


def run(held_out, r, seed):
    tag = f"r{r:g}__{held_out}__s{seed}"
    done = os.path.join(CKPT, tag + ".json")
    if os.path.exists(done):
        print(f"[o'tkazildi] {tag}"); return json.load(open(done))

    tr_set, va_set = split_cases({x["case"] for x in INDEX if x["src"] != held_out}, seed)
    tr = [x for x in INDEX if x["case"] in tr_set]
    va = [x for x in INDEX if x["case"] in va_set]
    te = [x for x in INDEX if x["src"] == held_out]
    assert not any(x["src"] == held_out for x in tr + va), "chetdagi manba sizdi!"
    print(f"\n=== r={r}  ({SPACING/r:.2f} mm)  chetda {held_out}  seed {seed} ===")
    print(f"  o'qitish {len(tr)}  validatsiya {len(va)}  tashqi {len(te)}")

    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    if dev.type == "cuda": torch.cuda.manual_seed_all(seed)
    model = UNet().to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    dl = DataLoader(SliceDS(tr, True, r, EPOCH_SLICES), batch_size=BATCH, shuffle=True,
                    num_workers=NUM_WORKERS, pin_memory=PIN, drop_last=True,
                    persistent_workers=NUM_WORKERS > 0)

    best, best_state = -1.0, None
    t0 = time.time()
    for ep in range(EPOCHS):
        model.train(); tot = n = 0
        for x, y in dl:
            x, y = x.to(dev), y.to(dev)
            opt.zero_grad(set_to_none=True)
            loss = criterion(_fwd(model, x), y)
            if AMP:
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss.backward(); opt.step()
            tot += loss.item() * x.size(0); n += x.size(0)
        sched.step()
        if (ep + 1) % EVAL_EVERY == 0 or ep == EPOCHS - 1:
            vd = val_dice(model, va, r); star = ""
            if vd > best:
                best, star = vd, " *"
                best_state = {kk: v.detach().clone() for kk, v in model.state_dict().items()}
            print(f"  ep {ep:>2}  loss {tot/max(n,1):.4f}  val {vd:.4f}{star}  "
                  f"{time.time()-t0:.0f}s", flush=True)

    model.load_state_dict(best_state)
    per = evaluate(model, te, r)
    g = lambda key: float(np.nanmean([v[key] for v in per.values()]))
    res = {"r": r, "spacing_mm": SPACING / r, "held_out": held_out, "seed": seed,
           "n_test": len(te), "dice": g("dice"), "assd_mm": g("assd_mm"),
           "gt_vox": g("gt_vox"), "img_std": g("img_std"), "sharp": g("sharp"),
           "internal_val": best, "per_case": per,
           "minutes": round((time.time()-t0)/60, 1)}
    json.dump(res, open(done, "w"), indent=2)
    print(f"  Dice {res['dice']:.4f}   ASSD {res['assd_mm']:.3f} mm   "
          f"o'tkirlik {res['sharp']:.3f}   o'sma {res['gt_vox']:.0f} voksel   "
          f"[{res['minutes']:.0f} daq]")
    del model, opt, dl, best_state; gc.collect()
    if dev.type == "cuda": torch.cuda.empty_cache()
    return res


results = []
t_all = time.time(); total = len(RES)*len(HELD_OUT)*len(SEEDS); done_n = 0
for seed in SEEDS:
    for ho in HELD_OUT:
        for r in RES:
            results.append(run(ho, r, seed)); done_n += 1
            el = (time.time()-t_all)/60
            print(f"  [{done_n}/{total}]  {el:.0f} daq, "
                  f"qolgan ~{el/done_n*(total-done_n):.0f} daq", flush=True)

## 6. Natija

In [ ]:
COLS = ("held_out","r","spacing_mm","seed","n_test","dice","assd_mm",
        "gt_vox","sharp","img_std","internal_val","minutes")
df = pd.DataFrame([{c: r[c] for c in COLS} for r in results]).sort_values(
        ["held_out","r"], ascending=[True, False])
df.to_csv(os.path.join(OUT, "ressweep.csv"), index=False)
display(df.round(4))

rows = []
for res in results:
    for key, v in res["per_case"].items():
        rows.append({"r": res["r"], "spacing_mm": res["spacing_mm"],
                     "held_out": res["held_out"], "seed": res["seed"],
                     "case": key, **v})
pc = pd.DataFrame(rows)
pc.to_csv(os.path.join(OUT, "ressweep_percase.csv"), index=False)
print(f"\nressweep_percase.csv: {len(pc)} qator")

# --- VALIDLIK: ground truth hamma r da bir xil bo'lishi SHART ---
print("\n" + "="*72)
print("VALIDLIK TEKSHIRUVI")
print("="*72)
bad = []
for (ho, seed, case), g in pc.groupby(["held_out", "seed", "case"]):
    if g.gt_vox.nunique() != 1:
        bad.append((ho, seed, case, sorted(g.gt_vox.unique())))
if bad:
    print(f"  XATO: {len(bad)} holatda ground truth r bilan o'zgargan!")
    for b in bad[:5]: print("   ", b)
else:
    print("  ground truth hamma r da aynan bir xil — vazifa o'zgarmagan [ok]")
    print("  qirqish yo'q, masshtab siljishi yo'q: faqat rezolyutsiya o'zgardi")

sh = df.groupby("r").sharp.mean().sort_index(ascending=False)
print(f"\n  O'tkirlik r bilan tushdimi? {', '.join(f'r={k:g}:{v:.3f}' for k, v in sh.items())}")
print(f"  {'[ok] monoton' if all(sh.values[i] >= sh.values[i+1] for i in range(len(sh)-1)) else '[XATO] monoton emas'}")

from scipy import stats
print("\n" + "="*72)
print("DOZA-JAVOB: rezolyutsiya -> Dice")
print("="*72)
for ho in HELD_OUT:
    s = df[df.held_out == ho].sort_values("r", ascending=False)
    if s.empty: continue
    print(f"\n  chetda {ho}:")
    print(f"    {'qadam':>8} {'Dice':>8} {'ASSD mm':>9} {'o`tkirlik':>10}")
    g = s.groupby("spacing_mm")[["dice", "assd_mm", "sharp"]].mean().sort_index()
    for sp_, rr in g.iterrows():
        print(f"    {sp_:>7.2f}m {rr.dice:>8.4f} {rr.assd_mm:>9.3f} {rr.sharp:>10.3f}")
    b = g.loc[g.index.min()]       # 1.5 mm
    print("    1.5 mm ga nisbatan: ", end="")
    for sp_, rr in g.iterrows():
        if sp_ == g.index.min(): continue
        print(f"{sp_:.1f}mm: {rr.dice-b.dice:+.4f}   ", end="")
    print()

print("\n  TAXMINLARNI TEKSHIRISH (holat darajasida, juftlashgan):")
for ho in HELD_OUT:
    sub = pc[pc.held_out == ho].dropna(subset=["dice"])
    if sub.empty: continue
    r1, p1 = stats.spearmanr(sub["spacing_mm"], sub["dice"])
    print(f"\n    chetda {ho}:")
    print(f"      1) Dice qadam bilan tushadimi?  rho = {r1:+.3f}  p = {p1:.2e}"
          f"   {'TASDIQ' if (r1 < -0.1 and p1 < 0.05) else 'RAD'}")
    w = sub.pivot_table(index=["case","seed"], columns="spacing_mm", values="dice")
    lo, hi = w.columns.min(), w.columns.max()
    a = w[[lo, hi]].dropna()
    if len(a) > 5:
        t = stats.wilcoxon(a[lo], a[hi])
        d_ = a[hi].mean() - a[lo].mean()
        print(f"      {lo:.1f}mm vs {hi:.1f}mm juftlashgan: "
              f"{a[lo].mean():.4f} -> {a[hi].mean():.4f}  ({d_:+.4f})  "
              f"p = {t.pvalue:.2e}  n={len(a)}")
        print(f"      2) tushish mo'tadilmi (<0.03)?  "
              f"{'TASDIQ' if abs(d_) < 0.03 else 'RAD'}")

print("""
  O'QISH:
  Dice 4.5 mm gacha sezilarli tushsa — rezolyutsiya ahamiyatli, va
  V0 ning manbalararo effektida u rol o'ynashi mumkin (V0 DUKE ni
  ~2.3x qo'pollashtiradi).

  Sezilarli tushmasa — rezolyutsiya ham mexanizm emas. Masshtab (09),
  ko'rish maydoni (09, qirqish yo'q bu yerda) va rezolyutsiya uchalasi
  ham rad etilgan bo'ladi; qoladigan nomzod piksel fazosidagi domen
  o'xshashligi. Maqolada uchala rad etilgan gipoteza ham yoziladi —
  bu kuchsizlik emas, o'lchangan bilim.""")

## 7. Keyingi qadam

`ressweep.csv` va `ressweep_percase.csv` ni yuboring.